In [2]:
import os
import cv2
import json
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\hesha\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
VIDEO_PATH = "Mindblowing.mp4"

In [4]:
MODEL_PATH = "best.pt"
custom_model = YOLO(MODEL_PATH)
baseline_model = YOLO('yolov8n-pose.pt')

## Multiple Player Tracking With Both Fine-Tuned & Baseline Model Combination

In [10]:
# ──────────────────────────────────────────────────────────────────────────────
# MULTIPLE PLAYER TRACKING WITH BOTH FINE-TUNED & BASELINE MODEL COMBINATION
# (Single-player target locking applied — correct keypoints via baseline model)
# ──────────────────────────────────────────────────────────────────────────────

# --- CONFIG ---
OUTPUT_VIDEO = "analyzed_single_player_badminton.mp4"
OUTPUT_JSON  = "analyzed_single_player_badminton.json"

CONF_THRES          = 0.35
KEYPOINT_CONF_THRES = 0.25

MAX_MISSING_FRAMES  = 40
MAX_CENTER_DISTANCE = 200
MIN_MATCH_SCORE     = 0.25


COURT_Y_MIN = 0.28   # feet must be below 28% of frame height
COURT_Y_MAX = 0.62   # feet must be above 62% of frame height  (excludes near player)

# Motion detection: a candidate must have at least this fraction of its crop
# covered by moving pixels (frame-diff) to be considered an active player.
MOTION_THRESHOLD = 0.04   # 4% of crop pixels must be moving

KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

SKELETON = [
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 6),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (0, 1), (0, 2),
    (1, 3), (2, 4)
]

# ── Helper functions ───────────────────────────────────────────────────────────

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

def bbox_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])

def center_distance(boxA, boxB):
    return np.linalg.norm(bbox_center(boxA) - bbox_center(boxB))

def bbox_area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)

def safe_crop(frame, box):
    h, w = frame.shape[:2]
    x1 = int(max(0, min(w - 1, box[0])))
    y1 = int(max(0, min(h - 1, box[1])))
    x2 = int(max(0, min(w - 1, box[2])))
    y2 = int(max(0, min(h - 1, box[3])))
    if x2 <= x1 or y2 <= y1:
        return None
    return frame[y1:y2, x1:x2]

def color_histogram(frame, box):
    crop = safe_crop(frame, box)
    if crop is None or crop.size == 0:
        return None
    crop = cv2.resize(crop, (64, 128))
    hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [32, 32], [0, 180, 0, 256])
    cv2.normalize(hist, hist)
    return hist

def histogram_similarity(histA, histB):
    if histA is None or histB is None:
        return 0.0
    return max(0.0, min(1.0, cv2.compareHist(histA, histB, cv2.HISTCMP_CORREL)))

def motion_score(frame, prev_frame, box):
    """Fraction of bbox crop pixels that changed significantly between frames."""
    if prev_frame is None:
        return 1.0   # no prior frame → don't penalise on first lock
    crop_curr = safe_crop(frame,      box)
    crop_prev = safe_crop(prev_frame, box)
    if crop_curr is None or crop_prev is None:
        return 0.0
    if crop_curr.shape != crop_prev.shape:
        crop_prev = cv2.resize(crop_prev, (crop_curr.shape[1], crop_curr.shape[0]))
    diff  = cv2.absdiff(crop_curr, crop_prev)
    gray  = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    moved = np.count_nonzero(gray > 25)          # pixels that changed by >25 intensity
    return moved / (gray.size + 1e-6)

def is_inside_court(box, frame_h):
    """
    True only if the detection's feet (bbox bottom) fall within the far-player
    court band. Rejects spectators above the back wall AND the near-side player.
    """
    foot_y = box[3] / frame_h          # normalised y of bbox bottom edge
    return COURT_Y_MIN < foot_y < COURT_Y_MAX

def face_visibility_score(kpt_conf):
    score  = sum(2 for i in [0, 1, 2, 3, 4] if kpt_conf[i] > KEYPOINT_CONF_THRES)
    score += sum(1 for i in [5, 6]           if kpt_conf[i] > KEYPOINT_CONF_THRES)
    return score

def extract_detections(result, frame):
    detections = []
    if result.boxes is None or result.keypoints is None:
        return detections
    boxes     = result.boxes.xyxy.cpu().numpy()
    det_confs = result.boxes.conf.cpu().numpy()
    kpts_xy   = result.keypoints.xy.cpu().numpy()
    kpts_conf = result.keypoints.conf.cpu().numpy() \
                if result.keypoints.conf is not None \
                else np.ones((len(boxes), 17))
    for i in range(len(boxes)):
        if det_confs[i] < CONF_THRES:
            continue
        detections.append({
            "box":           boxes[i].tolist(),
            "det_conf":      float(det_confs[i]),
            "keypoints":     kpts_xy[i].tolist(),
            "keypoint_conf": kpts_conf[i].tolist(),
            "hist":          color_histogram(frame, boxes[i]),
        })
    return detections

def select_initial_target(detections, frame, prev_frame, frame_h):
    """
    Pick the best candidate using three hard filters then a score:
      HARD 1 — feet must be inside the far-player court band (COURT_Y_MIN..COURT_Y_MAX)
      HARD 2 — must show meaningful motion vs previous frame (not a static spectator)
    SCORE   — face visibility (primary) + detection confidence + bbox size
    """
    best_det, best_score = None, -1

    for det in detections:
        box      = det["box"]
        kpt_conf = np.array(det["keypoint_conf"])

        # Hard filter 1: court boundary
        if not is_inside_court(box, frame_h):
            continue

        # Hard filter 2: must be moving
        mv = motion_score(frame, prev_frame, box)
        if mv < MOTION_THRESHOLD:
            continue

        score = (
            face_visibility_score(kpt_conf)                  * 4.0 +
            det["det_conf"]                                  * 2.0 +
            min(bbox_area(box) / (frame_h * frame_h), 1.0)  * 1.0 +
            min(mv / 0.20, 1.0)                              * 1.0   # more motion = better
        )
        if score > best_score:
            best_score, best_det = score, det

    return best_det

def match_locked_target(detections, frame, prev_frame, locked_box, locked_hist, frame_h):
    """
    Re-identify the locked player each frame.
    Hard filter: still must be inside the court band.
    Scoring: IoU + distance + appearance + motion.
    """
    best_det, best_score = None, -1

    for det in detections:
        box = det["box"]

        if not is_inside_court(box, frame_h):
            continue

        iou  = compute_iou(locked_box, box)
        dist = center_distance(locked_box, box)
        if dist > MAX_CENTER_DISTANCE and iou < 0.05:
            continue

        area_ratio  = bbox_area(box) / (bbox_area(locked_box) + 1e-6)
        mv          = motion_score(frame, prev_frame, box)

        match_score = (
            iou                                              * 4.0 +
            max(0.0, 1.0 - dist / MAX_CENTER_DISTANCE)      * 2.0 +
            histogram_similarity(locked_hist, det["hist"])   * 3.0 +
            det["det_conf"]                                  * 1.0 +
            (1.0 - min(abs(1.0 - area_ratio), 1.0))         * 1.0
        ) / 11.0

        if match_score > best_score:
            best_score, best_det = match_score, det

    return (None, best_score) if best_score < MIN_MATCH_SCORE else (best_det, best_score)

def draw_pose(frame, kpts, kpt_conf):
    for p1, p2 in SKELETON:
        if kpt_conf[p1] > KEYPOINT_CONF_THRES and kpt_conf[p2] > KEYPOINT_CONF_THRES:
            cv2.line(frame,
                     (int(kpts[p1][0]), int(kpts[p1][1])),
                     (int(kpts[p2][0]), int(kpts[p2][1])),
                     (0, 255, 255), 2)
    for i, (x, y) in enumerate(kpts):
        if kpt_conf[i] > KEYPOINT_CONF_THRES:
            cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1)

## Use Inference Function

In [11]:

# ── Main processing loop ───────────────────────────────────────────────────────

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "Cannot open video"

frame_w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_w, frame_h))

tracking_json = {
    "video_name":      VIDEO_PATH,
    "baseline_model":  "yolov8n-pose.pt",
    "custom_model":    "best.pt (badminton fine-tuned)",
    "tracking_method": "baseline keypoints + court boundary + motion filter + appearance lock",
    "court_y_min":     COURT_Y_MIN,
    "court_y_max":     COURT_Y_MAX,
    "fps":             fps,
    "frame_width":     frame_w,
    "frame_height":    frame_h,
    "frames":          []
}

# Tracking state
locked              = False
locked_box          = None
locked_hist         = None
missing_count       = 0
last_good_detection = None
pose_name           = "Detecting..."
prev_frame          = None          # for motion detection

print(f"Processing {total_frames} frames...")

for frame_id in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret:
        break

    timestamp = frame_id / fps

    # ── Step 1: baseline model → correct keypoints for ALL detected people ──
    base_result = baseline_model(frame, conf=CONF_THRES, verbose=False)[0]
    detections  = extract_detections(base_result, frame)

    # ── Step 2: custom model → shot classification (frame-level) ───────────
    custom_result = custom_model(frame, verbose=False)[0]
    if custom_result.boxes and len(custom_result.boxes) > 0:
        pose_id   = int(custom_result.boxes.cls[0].item())
        pose_name = custom_result.names[pose_id]
        shot_conf = float(custom_result.boxes.conf[0].item())
    else:
        shot_conf = 0.0

    # ── Step 3: lock onto / track the single target player ─────────────────
    selected    = None
    match_score = None

    if not locked:
        if detections:
            selected = select_initial_target(detections, frame, prev_frame, frame_h)
            if selected is not None:
                locked              = True
                locked_box          = selected["box"]
                locked_hist         = selected["hist"]
                last_good_detection = selected
                missing_count       = 0
    else:
        selected, match_score = match_locked_target(
            detections, frame, prev_frame, locked_box, locked_hist, frame_h
        )
        if selected is not None:
            locked_box = selected["box"]
            new_hist   = selected["hist"]
            if locked_hist is not None and new_hist is not None:
                locked_hist = cv2.addWeighted(locked_hist, 0.85, new_hist, 0.15, 0)
            last_good_detection = selected
            missing_count       = 0
        else:
            missing_count += 1
            if missing_count <= MAX_MISSING_FRAMES:
                selected = last_good_detection
            else:
                locked = False
                locked_box = locked_hist = last_good_detection = None
                missing_count = 0

    # ── Step 4: annotate frame ──────────────────────────────────────────────
    frame_data = {
        "frame_id":            frame_id,
        "timestamp":           round(timestamp, 4),
        "player_detected":     False,
        "tracking_status":     "missing",
        "match_score":         match_score,
        "shot_classification": pose_name,
        "shot_confidence":     shot_conf,
        "bounding_box":        None,
        "detection_confidence": None,
        "keypoints":           []
    }

    if selected is not None:
        box     = selected["box"]
        kpts    = np.array(selected["keypoints"])
        kpt_conf = np.array(selected["keypoint_conf"])

        x1, y1, x2, y2 = map(int, box)
        status    = "tracked" if missing_count == 0 else "temporarily_predicted"
        box_color = (0, 255, 0)  if missing_count == 0 else (0, 165, 255)

        # Bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 3)

        # Labels
        cv2.putText(frame,
                    f"Target Player | {status}",
                    (x1, max(30, y1 - 30)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, box_color, 2)
        cv2.putText(frame,
                    f"Shot: {pose_name} ({shot_conf:.2f})",
                    (x1, max(55, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

        # Skeleton (from baseline model — correct keypoints)
        draw_pose(frame, kpts, kpt_conf)

        # JSON
        frame_data["player_detected"]      = True
        frame_data["tracking_status"]      = status
        frame_data["detection_confidence"] = float(selected["det_conf"])
        frame_data["bounding_box"] = {
            "x1": float(box[0]), "y1": float(box[1]),
            "x2": float(box[2]), "y2": float(box[3]),
            "width":  float(box[2] - box[0]),
            "height": float(box[3] - box[1]),
        }
        for idx, name in enumerate(KEYPOINT_NAMES):
            frame_data["keypoints"].append({
                "id":         idx,
                "name":       name,
                "x":          float(kpts[idx][0]),
                "y":          float(kpts[idx][1]),
                "confidence": float(kpt_conf[idx]),
            })

    tracking_json["frames"].append(frame_data)
    out.write(frame)
    prev_frame = frame.copy()

cap.release()
out.release()

Processing 481 frames...


 99%|█████████▉| 478/481 [02:29<00:00,  3.19it/s]


In [12]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(tracking_json, f, indent=4)

print("✅ Done")
print("Video:", OUTPUT_VIDEO)
print("JSON: ", OUTPUT_JSON)

✅ Done
Video: analyzed_single_player_badminton.mp4
JSON:  analyzed_single_player_badminton.json
